# Step 2a: Negative Intent Detection

This notebook analyzes search queries to identify which ones contain negative intent (user wants to EXCLUDE certain attributes).

**Output**: A checkpoint file mapping queries to their negative intent attributes.
**Next Step**: Use 02b_llm_evaluation_pairs.ipynb to evaluate query-product pairs.

In [1]:
import pandas as pd
import json
import os
import boto3
import time
from datetime import datetime
import logging
from tqdm import tqdm

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [2]:
%env AWS_PROFILE = worker_discovery_dev

env: AWS_PROFILE=worker_discovery_dev


## Configuration

In [3]:
# Configuration
SEARCH_RESULTS_DIR = "./search_results/"
NEGATIVE_INTENT_DIR = "./llm_evaluations/negative_intent"
os.makedirs(NEGATIVE_INTENT_DIR, exist_ok=True)

# AWS Bedrock configuration  
model_id = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"

# Initialize Bedrock client
bedrock = boto3.client('bedrock-runtime', region_name='us-east-1')

print(f"Search results directory: {SEARCH_RESULTS_DIR}")
print(f"Negative intent directory: {NEGATIVE_INTENT_DIR}")
print(f"Bedrock model: {model_id}")

Search results directory: ./search_results/
Negative intent directory: ./llm_evaluations/negative_intent
Bedrock model: us.anthropic.claude-3-7-sonnet-20250219-v1:0


## Load Search Results

In [4]:
def load_all_search_results():
    """Load all search result CSV files from all subdirectories and combine into structured data"""
    
    # Check if directory exists
    if not os.path.exists(SEARCH_RESULTS_DIR):
        logger.error(f"Directory not found: {SEARCH_RESULTS_DIR}")
        return pd.DataFrame()
    
    all_results = []
    csv_files = []
    
    # Walk through all subdirectories to find CSV files
    for root, dirs, files in os.walk(SEARCH_RESULTS_DIR):
        for file in files:
            if file.endswith('.csv'):
                full_path = os.path.join(root, file)
                # Get relative path from search_results directory
                rel_path = os.path.relpath(full_path, SEARCH_RESULTS_DIR)
                csv_files.append((full_path, rel_path))
    
    print(f"Search directory: {SEARCH_RESULTS_DIR}")
    print(f"Found {len(csv_files)} CSV files:")
    for full_path, rel_path in csv_files:
        print(f"  {rel_path}")
    
    # Load each CSV file
    for full_path, rel_path in tqdm(csv_files, desc="Loading CSV files"):
        try:
            df = pd.read_csv(full_path)
            df['source_file'] = rel_path  # Include subdirectory in source file name
            df['subdirectory'] = os.path.dirname(rel_path) if os.path.dirname(rel_path) else 'root'
            all_results.append(df)
                    
        except Exception as e:
            logger.error(f"Error loading {rel_path}: {e}")
            continue
    
    # Combine all CSV files into one DataFrame
    if all_results:
        combined_df = pd.concat(all_results, ignore_index=True)
        return combined_df
    else:
        return pd.DataFrame()

# Load all results
df_all_results = load_all_search_results()
print(f"\nLoaded {len(df_all_results)} total search result entries")
if len(df_all_results) > 0:
    print(f"Unique search terms: {df_all_results['search_term'].nunique()}")
    print(f"Unique products: {df_all_results['part_number'].nunique()}")
    print(f"Endpoint distribution:")
    print(df_all_results['endpoint_type'].value_counts())
    print(f"Subdirectory distribution:")
    print(df_all_results['subdirectory'].value_counts())
else:
    print("No data loaded. Please check that CSV files exist in subdirectories of search_results/")

Search directory: ./search_results/
Found 8 CSV files:
  old_control_before_revenue_func/l1_hybrid_search_results.csv
  old_control_before_revenue_func/l2_search_results.csv
  old_control_before_revenue_func/knn_search_results.csv
  old_control_before_revenue_func/lexical_search_results.csv
  control/l1_hybrid_search_results.csv
  control/l2_search_results.csv
  control/knn_search_results.csv
  control/lexical_search_results.csv


Loading CSV files: 100%|██████████| 8/8 [00:00<00:00, 22.40it/s]


Loaded 277497 total search result entries
Unique search terms: 1000
Unique products: 19344
Endpoint distribution:
endpoint_type
knn          72000
l1_hybrid    71383
l2           71372
lexical      62742
Name: count, dtype: int64
Subdirectory distribution:
subdirectory
control                            138800
old_control_before_revenue_func    138697
Name: count, dtype: int64


## Extract Unique Queries

In [5]:
# Extract unique search queries for negative intent analysis
unique_queries = df_all_results['search_term'].unique()
print(f"Found {len(unique_queries)} unique search queries to analyze")

# Show sample queries
print(f"\nSample queries:")
for i, query in enumerate(unique_queries[:10]):
    print(f"  {i+1}. '{query}'")

Found 1000 unique search queries to analyze

Sample queries:
  1. 'chicken free dry dog food'
  2. 'grain free dog food'
  3. 'grain free dog treats'
  4. 'grain free dry cat food'
  5. 'grain free wet cat food'
  6. 'no hide chews'
  7. 'no pull dog harnesses'
  8. 'pork chomps rawhide free'
  9. 'rawhide free dog bones'
  10. 'stuffing free dog toys'


## Negative Intent Detection

In [6]:
def detect_negative_intent_with_llm(query):
    """Use LLM to detect if query has negative intent and extract it"""
    prompt = f"""
You are analyzing search queries from Chewy.com, a pet product retailer. Users search for pet food, toys, supplies, and other pet-related items.

Analyze this search query to determine if it contains negative intent (user wants to EXCLUDE certain attributes or ingredients).

QUERY: "{query}"

If the query contains negative intent, respond with the specific attribute/ingredient the user wants to exclude.
If no negative intent is found, respond with "NONE".

Examples:
- "grain free dog food" → "grain"
- "no pull dog harness" → "pull" 
- "rawhide free treats" → "rawhide"
- "dog food" → "NONE"

RESPONSE (only the excluded attribute or NONE):"""
    
    try:
        # Check if bedrock client exists
        if 'bedrock' not in globals() or bedrock is None:
            raise Exception("Bedrock client not initialized. Run the configuration cells first.")
            
        body = {
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 20,
            "messages": [{"role": "user", "content": prompt}]
        }
        
        response = bedrock.invoke_model(
            modelId=model_id,
            body=json.dumps(body)
        )
        
        response_data = json.loads(response['body'].read())
        result = response_data['content'][0]['text'].strip().lower()
        return None if result == "none" else result
        
    except Exception as e:
        logger.error(f"Error detecting negative intent for '{query}': {e}")
        return None

def run_negative_intent_detection(unique_queries):
    """Run negative intent detection with checkpointing"""
    
    # Check for existing negative intent results
    negative_intent_file = os.path.join(NEGATIVE_INTENT_DIR, 'negative_intent_detection.json')
    existing_results = {}
    
    if os.path.exists(negative_intent_file):
        with open(negative_intent_file, 'r') as f:
            existing_results = json.load(f)
        print(f"Found existing negative intent results for {len(existing_results)} queries")
    
    # Filter out already processed queries
    remaining_queries = [q for q in unique_queries if q not in existing_results]
    print(f"Processing {len(remaining_queries)} remaining queries out of {len(unique_queries)} total")
    
    if len(remaining_queries) == 0:
        print("✅ All queries already processed for negative intent detection!")
        return existing_results
    
    # Process remaining queries
    processed_count = 0
    for query in tqdm(remaining_queries, desc="Analyzing queries for negative intent"):
        intent = detect_negative_intent_with_llm(query)
        existing_results[query] = intent
        processed_count += 1
        
        # Save checkpoint every 20 queries
        if processed_count % 20 == 0:
            with open(negative_intent_file, 'w') as f:
                json.dump(existing_results, f, indent=2)
            logger.info(f"Checkpoint saved: {len(existing_results)} queries processed")
        
        time.sleep(0.1)  # Rate limiting
    
    # Final save
    with open(negative_intent_file, 'w') as f:
        json.dump(existing_results, f, indent=2)
    
    return existing_results

# Run negative intent detection
print("=== NEGATIVE INTENT DETECTION ===")
negative_intent_map = run_negative_intent_detection(unique_queries)

=== NEGATIVE INTENT DETECTION ===
Found existing negative intent results for 1000 queries
Processing 0 remaining queries out of 1000 total
✅ All queries already processed for negative intent detection!


In [7]:
negative_intent_map

{'chicken free dry dog food': 'chicken',
 'grain free dog food': 'grain',
 'grain free dog treats': 'grain',
 'grain free dry cat food': 'grain',
 'grain free wet cat food': 'grain',
 'no hide chews': 'hide',
 'no pull dog harnesses': 'pull',
 'pork chomps rawhide free': 'rawhide',
 'rawhide free dog bones': 'rawhide',
 'stuffing free dog toys': 'stuffing',
 '2 hounds design freedom no pull dog harness': 'pull',
 'acana grain free': 'grain',
 'acana grain free cat': 'grain',
 'acana grain-free food for dogs': 'grain',
 'adjustable no pull harness': 'pull',
 'aller free': 'aller',
 'allergy free cat food': 'allergy',
 'allergy free dog food': 'allergy',
 'allergy free dog treats': 'allergy',
 'american journey grain free': 'grain',
 'american journey grain free dog food': 'grain',
 'amino acid': None,
 'amino acid horse': None,
 'amino acids': None,
 'amino b plex': None,
 'amino b-plex': None,
 'audubon park waste free': 'waste free',
 'bark collar no shock': 'shock',
 'bark no more': 

## Analysis Results

In [8]:
# Analyze results
queries_with_negative_intent = {k: v for k, v in negative_intent_map.items() if v is not None}

print(f"=== DETECTION RESULTS ===")
print(f"Total queries analyzed: {len(negative_intent_map)}")
print(f"Queries with negative intent: {len(queries_with_negative_intent)}")
print(f"Queries without negative intent: {len(negative_intent_map) - len(queries_with_negative_intent)}")
print(f"Detection rate: {len(queries_with_negative_intent) / len(negative_intent_map) * 100:.1f}%")

if len(queries_with_negative_intent) > 0:
    # Show distribution of negative intent types
    intent_values = list(queries_with_negative_intent.values())
    intent_counts = pd.Series(intent_values).value_counts()
    
    print(f"\n=== NEGATIVE INTENT TYPES ===")
    print(intent_counts.head(10))
    
    # Print ALL unique negative intent values for review
    print(f"\n=== ALL UNIQUE NEGATIVE INTENT VALUES ({len(intent_counts)} total) ===")
    for intent_type in sorted(intent_counts.index):
        count = intent_counts[intent_type]
        # Check for potential issues
        issues = []
        if '\n' in intent_type:
            issues.append("⚠️ MULTI-LINE")
        if len(intent_type.split()) > 3:
            issues.append("⚠️ LONG")
        
        issue_str = ' '.join(issues) if issues else ''
        print(f"  '{intent_type}' (count={count}) {issue_str}")
    
    print(f"\n=== EXAMPLES WITH NEGATIVE INTENT ===")
    for intent_type in intent_counts.head(5).index:
        example_queries = [k for k, v in queries_with_negative_intent.items() if v == intent_type]
        print(f"\nIntent '{intent_type}' examples:")
        for query in example_queries[:3]:
            print(f"  - '{query}'")

# Show examples without negative intent
queries_without_negative_intent = [k for k, v in negative_intent_map.items() if v is None]
if len(queries_without_negative_intent) > 0:
    print(f"\n=== EXAMPLES WITHOUT NEGATIVE INTENT ===")
    print(f"Showing {min(10, len(queries_without_negative_intent))} examples:")
    for query in queries_without_negative_intent[:10]:
        print(f"  - '{query}'")

=== DETECTION RESULTS ===
Total queries analyzed: 1000
Queries with negative intent: 795
Queries without negative intent: 205
Detection rate: 79.5%

=== NEGATIVE INTENT TYPES ===
grain       332
chicken      59
pull         53
rawhide      47
hide         36
stuffing     24
dust         14
mess         14
gluten       10
odor         10
Name: count, dtype: int64

=== ALL UNIQUE NEGATIVE INTENT VALUES (102 total) ===
  'additives' (count=1) 
  'aller' (count=1) 
  'allergy' (count=3) 
  'bark' (count=1) 
  'bell' (count=1) 
  'bpa' (count=1) 
  'carpet' (count=1) 
  'cat pee' (count=1) 
  'catnip' (count=2) 
  'chew' (count=5) 
  'chicken' (count=59) 
  'choke' (count=1) 
  'clay' (count=2) 
  'clump' (count=1) 
  'clumping' (count=2) 
  'corn' (count=9) 
  'corn and soy' (count=1) 
  'd3' (count=1) 
  'dehp' (count=1) 
  'dust' (count=14) 
  'dust, scent' (count=1) 
  'ears' (count=2) 
  'escape' (count=1) 
  'fat' (count=2) 
  'fish' (count=1) 
  'fragrance' (count=3) 
  'free' (count

In [11]:
# Show negative intents with their corresponding search terms
print(f"\n=== NEGATIVE INTENTS WITH SEARCH TERMS ===")
print(f"(Showing intents with 5+ search terms)")

# Group queries by negative intent
intent_to_queries = {}
for query, intent in queries_with_negative_intent.items():
    if intent not in intent_to_queries:
        intent_to_queries[intent] = []
    intent_to_queries[intent].append(query)

# Sort by number of queries (descending)
sorted_intents = sorted(intent_to_queries.items(), key=lambda x: len(x[1]), reverse=True)

# Print intents with many queries
for intent, queries in sorted_intents:
    if len(queries) >= 5:  # Show intents with 5+ queries
        print(f"\n'{intent}' ({len(queries)} queries):")
        for query in queries[:5]:  # Show first 5 queries
            print(f"  - {query}")
        if len(queries) > 5:
            print(f"  ... and {len(queries) - 5} more")

print(f"\n=== SUMMARY ===")
print(f"Intents with 5+ queries: {sum(1 for _, queries in sorted_intents if len(queries) >= 5)}")
print(f"Intents with <5 queries: {sum(1 for _, queries in sorted_intents if len(queries) < 5)}")


=== NEGATIVE INTENTS WITH SEARCH TERMS ===
(Showing intents with 5+ search terms)

'grain' (332 queries):
  - grain free dog food
  - grain free dog treats
  - grain free dry cat food
  - grain free wet cat food
  - acana grain free
  ... and 327 more

'chicken' (59 queries):
  - chicken free dry dog food
  - blue buffalo chicken free
  - blue buffalo no chicken
  - cat food no chicken
  - cat treats no chicken
  ... and 54 more

'pull' (53 queries):
  - no pull dog harnesses
  - 2 hounds design freedom no pull dog harness
  - adjustable no pull harness
  - best no pull harness
  - dog harness large no pull
  ... and 48 more

'rawhide' (47 queries):
  - pork chomps rawhide free
  - rawhide free dog bones
  - chew bones rawhide free
  - chews no rawhide
  - country kitchen rawhide free twists
  ... and 42 more

'hide' (36 queries):
  - no hide chews
  - canine naturals hide free
  - earth animal no hide
  - earth animal no hide chews
  - earth animal no hide chews large
  ... and 31 mo

## Manual Fixes

In [10]:
# Apply manual fixes to clean up negative intent values
def apply_manual_fixes(intent_map):
    """Apply manual fixes to clean up problematic negative intent values"""
    
    fixed_map = intent_map.copy()
    fixes_applied = []
    
    # Define manual fix rules
    fix_rules = {
        # 1. Fix multi-line/verbose LLM responses
        "non clumping unscented cat litter\n\nthe user is actually looking for cat litter": "clumping",
        "unscented\ndust": "dust, scent",
        
        # 2. Consolidate similar intents (remove "non", "no", "free" prefixes)
        "waste free": "waste",
        "no melt": "melt",
        "non clumping": "clumping",
        "non absorbent": "absorbent",
        "non gmo": "gmo",
        "no bark": "bark",
        "non prescription": "prescription",
        "cat pee": "pee",
        "poop eating": "poop",
        "scented": "scent",
        "unscented": "scent",
        "stuffed": "stuffing",
        
        # 3. Consolidate word order differences
        "soy and corn": "corn, soy",
        "corn and soy": "corn, soy",
        
        # 4. Fix typos/unclear terms
        "gain": "grain",  # Likely typo
        "stuff": "stuffing",
        "aller": "allergy",  # 
        
        # 5. Too generic - set to None
        "free": None,
        "holes": None,  # Too vague
        "hides": None,  # Confusing with "hide"
        
        # 6. Consolidate duplicates
        "squeaky": "squeak", # non squeaky dog toys
        "track": "tracking",
        "toot": None,  # Too vague/rare
    }
    
    # Apply fixes
    for query, intent in list(fixed_map.items()):
        if intent in fix_rules:
            new_intent = fix_rules[intent]
            fixed_map[query] = new_intent
            fixes_applied.append((query, intent, new_intent))
    
    return fixed_map, fixes_applied

# Apply fixes
negative_intent_map_fixed, fixes = apply_manual_fixes(negative_intent_map)

# Report fixes
print(f"=== MANUAL FIXES APPLIED ===")
print(f"Total fixes: {len(fixes)}")
if fixes:
    print(f"\nFixed mappings (showing all):")
    for query, old_intent, new_intent in sorted(fixes, key=lambda x: (x[1] or '', x[2] or '')):
        new_str = f"'{new_intent}'" if new_intent else "None"
        print(f"  '{old_intent}' → {new_str}")
    
    # Group by fix type
    consolidations = [f for f in fixes if f[2] is not None and f[1] != f[2]]
    removals = [f for f in fixes if f[2] is None]
    print(f"\n  Consolidations: {len(consolidations)}")
    print(f"  Removals (set to None): {len(removals)}")

# Save manually fixed version
manual_fix_file = os.path.join(NEGATIVE_INTENT_DIR, 'negative_intent_detection_manual_fix.json')
with open(manual_fix_file, 'w') as f:
    json.dump(negative_intent_map_fixed, f, indent=2)

print(f"\n✓ Manually fixed version saved to: {manual_fix_file}")
print(f"\nOriginal file (unchanged): {os.path.join(NEGATIVE_INTENT_DIR, 'negative_intent_detection.json')}")
print(f"Manual fix file (cleaned): {manual_fix_file}")

# Show unique values after fixes
fixed_intents = [v for v in negative_intent_map_fixed.values() if v is not None]
unique_fixed = sorted(set(fixed_intents))
print(f"\n=== UNIQUE NEGATIVE INTENTS AFTER FIXES ===")
print(f"Total unique: {len(unique_fixed)}")
print(f"Values: {unique_fixed}")

=== MANUAL FIXES APPLIED ===
Total fixes: 37

Fixed mappings (showing all):
  'aller' → 'allergy'
  'cat pee' → 'pee'
  'corn and soy' → 'corn, soy'
  'free' → None
  'free' → None
  'gain' → 'grain'
  'hides' → None
  'holes' → None
  'no bark' → 'bark'
  'no melt' → 'melt'
  'non absorbent' → 'absorbent'
  'non clumping' → 'clumping'
  'non clumping' → 'clumping'
  'non clumping' → 'clumping'
  'non clumping' → 'clumping'
  'non clumping' → 'clumping'
  'non clumping' → 'clumping'
  'non clumping' → 'clumping'
  'non clumping unscented cat litter

the user is actually looking for cat litter' → 'clumping'
  'non gmo' → 'gmo'
  'non gmo' → 'gmo'
  'non gmo' → 'gmo'
  'non prescription' → 'prescription'
  'non prescription' → 'prescription'
  'poop eating' → 'poop'
  'scented' → 'scent'
  'soy and corn' → 'corn, soy'
  'squeaky' → 'squeak'
  'stuff' → 'stuffing'
  'stuffed' → 'stuffing'
  'toot' → None
  'track' → 'tracking'
  'unscented' → 'scent'
  'unscented' → 'scent'
  'unscented' 